# Federated Learning Demo on `multi_PRS_dataset.csv`

This notebook compares three settings on the 3-feature PRS dataset:

1. Independent site models
2. Federated GLORE-style logistic regression
3. Centralized pooled training

Features:
- `CAD_PRS_z`
- `BMI_PRS_z`
- `HTN_PRS_z`

Label:
- `CAD_status`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from federated_glore import (
    FederatedLogisticRegression,
    LocalSite,
    calibration_table,
    evaluate_predictions,
    split_sites,
)

In [ ]:
data_path = Path('multi_PRS_dataset.csv')
if not data_path.exists():
    raise FileNotFoundError('Could not find multi_PRS_dataset.csv in the current folder.')

df = pd.read_csv(data_path)
df.head()

In [ ]:
feature_cols = ['CAD_PRS_z', 'BMI_PRS_z', 'HTN_PRS_z']
label_col = 'CAD_status'

print('Shape:', df.shape)
print('Label counts:')
print(df[label_col].value_counts())

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=88,
    stratify=df[label_col],
)

site_dfs = split_sites(train_df, n_sites=4, random_state=88, stratify_col=label_col)
sites = [LocalSite(site_id=i + 1, data=site_df.reset_index(drop=True), feature_cols=feature_cols, label_col=label_col) for i, site_df in enumerate(site_dfs)]

site_summary = pd.DataFrame(
    {
        'site_id': [site.site_id for site in sites],
        'n_samples': [len(site.data) for site in sites],
        'positive_rate': [site.data[label_col].mean() for site in sites],
    }
)
site_summary

In [ ]:
x_test = test_df[feature_cols]
y_test = test_df[label_col]

independent_rows = []
independent_probs = []

for site in sites:
    site_model = FederatedLogisticRegression().fit([site])
    site_prob = site_model.predict_proba(x_test)
    independent_probs.append(site_prob)
    independent_rows.append({'Model': f'Independent Site {site.site_id}', **evaluate_predictions(y_test.to_numpy(), site_prob)})

independent_results = pd.DataFrame(independent_rows)
independent_results

In [ ]:
fed_model = FederatedLogisticRegression().fit(sites)
central_model = FederatedLogisticRegression().fit([
    LocalSite(site_id=0, data=train_df.reset_index(drop=True), feature_cols=feature_cols, label_col=label_col)
])

fed_prob = fed_model.predict_proba(x_test)
central_prob = central_model.predict_proba(x_test)
independent_mean_prob = np.mean(np.column_stack(independent_probs), axis=1)

In [ ]:
results = pd.concat([
    independent_results,
    pd.DataFrame([
        {'Model': 'Independent Mean Ensemble', **evaluate_predictions(y_test.to_numpy(), independent_mean_prob)},
        {'Model': 'Federated GLORE', **evaluate_predictions(y_test.to_numpy(), fed_prob)},
        {'Model': 'Centralized pooled', **evaluate_predictions(y_test.to_numpy(), central_prob)},
    ]),
], ignore_index=True)

results

In [ ]:
coef_compare = pd.DataFrame(
    {
        'term': ['intercept'] + feature_cols,
        'federated': [fed_model.intercept_, *fed_model.coef_],
        'centralized': [central_model.intercept_, *central_model.coef_],
    }
)
coef_compare['abs_diff'] = (coef_compare['federated'] - coef_compare['centralized']).abs()
coef_compare

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(fed_model.history_['iter'], fed_model.history_['log_loss'], marker='o')
plt.title('Federated Training History')
plt.xlabel('Iteration')
plt.ylabel('Global Log Loss')
plt.tight_layout()
plt.show()

In [ ]:
calib_df = pd.concat([
    calibration_table(y_test.to_numpy(), independent_mean_prob, 'Independent Mean Ensemble'),
    calibration_table(y_test.to_numpy(), fed_prob, 'Federated GLORE'),
    calibration_table(y_test.to_numpy(), central_prob, 'Centralized pooled'),
], ignore_index=True)

plt.figure(figsize=(7, 5))
for model_name, group in calib_df.groupby('Model'):
    plt.plot(group['mean_pred'], group['obs_rate'], marker='o', label=model_name)

x = np.linspace(0, 1, 100)
plt.plot(x, x, linestyle='--', color='black', label='Perfect calibration')
plt.title('Calibration Plot')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Event Rate')
plt.legend()
plt.tight_layout()
plt.show()